# S08 · Streaming estructurado del **viento**: ventanas, watermarking, checkpoint y deduplicación
**Nayder Arce Mayta** · Proyecto ClimateRisk Andes · topic `viento-eventos` (3 particiones, key = `sensorId`)

**Antes de correr:** Kafka + `uso-viento` arriba, y el productor corriendo (ver README). Para tener
más eventos en la demo usa `INTERVALO_S=15`:
```
docker compose -f uso-viento/compose.yml exec -e INTERVALO_S=15 uso-viento python productor_openmeteo.py
```

In [1]:
import os, time, json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType, IntegerType

BASE = os.getenv("VIENTO_BASE", "/opt")
ART = f"{BASE}/artifacts"
KAFKA = os.getenv("KAFKA_BOOTSTRAP", "kafka:9092")
TOPIC = "viento-eventos"
PUSHGATEWAY = os.getenv("PUSHGATEWAY", "http://pushgateway:9091")

spark = (SparkSession.builder
         .appName("s08-streaming-viento-nayder")
         .master("local[*]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
         .config("spark.sql.shuffle.partitions", "3")
         .config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.ui.port", "4040")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "|", time.strftime("%Y-%m-%d %H:%M:%S"))

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6d0daa8f-0657-4efc-8366-63e1dde70e5c;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

Spark 4.2.0 | 2026-11-03 00:44:30


## 1. Lectura batch del topic (qué hay guardado en Kafka)

In [2]:
lote = (spark.read.format("kafka").option("kafka.bootstrap.servers", KAFKA)
        .option("subscribe", TOPIC).option("startingOffsets", "earliest").load())
print("mensajes en el topic:", lote.count())
lote.groupBy("partition").count().orderBy("partition").show()      # distribución por key/sensor
lote.selectExpr("CAST(key AS STRING) key", "CAST(value AS STRING) value", "partition", "offset") \
    .orderBy(F.desc("offset")).show(3, truncate=False)

mensajes en el topic: 565


+---------+-----+
|partition|count|
+---------+-----+
|        0|   87|
|        1|   88|
|        2|  390|
+---------+-----+

+--------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+
|key           |value                                                                                                                                                                                                                                                                                                                                |partition|offset|
+--------------+-----------------------------------------------------------------------------------------------------------------------------------------

## 2. Stream + parseo con esquema explícito

In [3]:
# Contrato viento.lectura v1 (docs/contratos.md)
esquema_viento = (StructType()
    .add("tipoEvento", StringType()).add("version", IntegerType())
    .add("sensorId", StringType()).add("estacion", StringType())
    .add("lat", DoubleType()).add("lon", DoubleType())
    .add("velocidadViento", DoubleType()).add("direccionViento", DoubleType()).add("rafaga", DoubleType())
    .add("temperatura", DoubleType()).add("humedad", DoubleType()).add("presion", DoubleType())
    .add("obsTime", LongType()).add("origen", StringType()).add("timestamp", LongType()))

crudo = (spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", KAFKA)
         .option("subscribe", TOPIC)
         .option("startingOffsets", "latest")
         .option("failOnDataLoss", "false")
         .load())

eventos = (crudo.select("partition", "offset", F.col("timestamp").alias("kafkaTs"),
                        F.col("value").cast("string").alias("json_str"))
           .select("partition", "offset", "kafkaTs", "json_str",
                   F.from_json("json_str", esquema_viento).alias("e"))
           .select("partition", "offset", "kafkaTs", "json_str", "e.*")
           .withColumn("ts", (F.col("timestamp") / 1000).cast("timestamp"))       # tiempo del evento (lectura)
           .withColumn("obs_ts", (F.col("obsTime") / 1000).cast("timestamp")))    # paso de 15 min del modelo
eventos.printSchema()

root
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTs: timestamp (nullable = true)
 |-- json_str: string (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- version: integer (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- estacion: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- velocidadViento: double (nullable = true)
 |-- direccionViento: double (nullable = true)
 |-- rafaga: double (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- presion: double (nullable = true)
 |-- obsTime: long (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- obs_ts: timestamp (nullable = true)



In [4]:
# Validación en el stream: esquema (from_json devuelve null) vs rango físico
validados = (eventos
    .withColumn("esquemaOk", F.col("sensorId").isNotNull() & F.col("velocidadViento").isNotNull()
                & F.col("rafaga").isNotNull() & F.col("obsTime").isNotNull() & F.col("timestamp").isNotNull())
    .withColumn("rangoOk", F.col("velocidadViento").between(0, 200) & F.col("rafaga").between(0, 250)
                & F.col("direccionViento").between(0, 360) & F.col("humedad").between(0, 100)))
limpios = validados.filter("esquemaOk AND rangoOk")

### 2.1 Consola: ver llegar las lecturas

In [5]:
# Momento 1: arrancar
consulta = (validados.select("partition", "offset", "sensorId", "velocidadViento", "rafaga", "obs_ts", "ts",
                             "esquemaOk", "rangoOk")
            .writeStream.outputMode("append").format("console").option("truncate", "false")
            .trigger(processingTime="10 seconds").start())

In [6]:
# Momento 2: detener
consulta.stop()

## 3. Persistencia en Parquet con **checkpoint** (tolerancia a fallos)

In [7]:
P_EVENTOS = f"{ART}/s08_viento_eventos"
CHK_EVENTOS = f"{ART}/chk_s08_eventos"
consulta_parquet = (limpios.drop("json_str").writeStream.outputMode("append").format("parquet")
                    .option("path", P_EVENTOS).option("checkpointLocation", CHK_EVENTOS)
                    .trigger(processingTime="10 seconds").start())

In [8]:
# (repetible) leer lo guardado mientras el stream sigue
g = spark.read.parquet(P_EVENTOS)
print("filas en Parquet:", g.count())
g.groupBy("sensorId").agg(F.count("*").alias("n"), F.countDistinct("obsTime").alias("obsTime_distintos")).show()

filas en Parquet: 36


[Stage 11:====>                                                   (1 + 11) / 12]

+-----------------+---+-----------------+
|         sensorId|  n|obsTime_distintos|
+-----------------+---+-----------------+
|juliaca-viento-01| 11|                2|
|  lampa-viento-03| 11|                2|
|   puno-viento-02| 11|                2|
|    viento-tardio|  1|                1|
| viento-duplicado|  2|                1|
+-----------------+---+-----------------+



> **Hallazgo propio:** `n` > `obsTime_distintos` porque consultamos la API cada 15–60 s pero el modelo
> Open-Meteo solo cambia cada 15 min → la misma lectura llega repetida. Eso se resuelve en la sección 6 (dedup).

## 4. Ventana **tumbling** de 1 min sin y con watermark

In [9]:
# Momento 1: SIN watermark (el estado crece para siempre: outputMode complete)
sin_limite = (limpios.groupBy(F.window("ts", "1 minute"), "sensorId")
              .agg(F.round(F.avg("velocidadViento"), 1).alias("vel_prom"), F.max("rafaga").alias("rafaga_max"),
                   F.count("*").alias("n")))
consulta = (sin_limite.writeStream.outputMode("complete").format("console").option("truncate", "false")
            .trigger(processingTime="15 seconds").start())

In [10]:
consulta.stop()

In [11]:
# Momento 1: CON watermark de 2 min (Spark puede descartar estado viejo; outputMode update)
con_wm = (limpios.withWatermark("ts", "2 minutes")
          .groupBy(F.window("ts", "1 minute"), "sensorId")
          .agg(F.round(F.avg("velocidadViento"), 1).alias("vel_prom"), F.max("rafaga").alias("rafaga_max"),
               F.count("*").alias("n")))
consulta = (con_wm.writeStream.outputMode("update").format("console").option("truncate", "false")
            .option("checkpointLocation", f"{ART}/chk_s08_wm")
            .trigger(processingTime="15 seconds").start())

[Stage 17:>                                                         (0 + 3) / 3]

In [12]:
consulta.stop()

## 5. Ventana **deslizante** (5 min cada 1 min): tendencia del viento por estación

In [13]:
deslizante = (limpios.withWatermark("ts", "2 minutes")
              .groupBy(F.window("ts", "5 minutes", "1 minute"), "estacion")
              .agg(F.round(F.avg("velocidadViento"), 1).alias("vel_prom_5min"), F.max("rafaga").alias("rafaga_max_5min")))
consulta = (deslizante.writeStream.outputMode("update").format("console").option("truncate", "false")
            .option("checkpointLocation", f"{ART}/chk_s08_desl")
            .trigger(processingTime="15 seconds").start())

In [14]:
consulta.stop()

## 6. Evento **tardío** vs watermark (se publica una lectura de prueba con 10 min de atraso)

In [15]:
import threading
from pyspark.sql.functions import to_json, struct

COLS = ["tipoEvento", "version", "sensorId", "estacion", "lat", "lon", "velocidadViento", "direccionViento", "rafaga",
        "temperatura", "humedad", "presion", "obsTime", "origen", "timestamp"]

def publicar(sensor, atraso_s, rafaga=99.0, veces=1):
    t = int(time.time() * 1000) - atraso_s * 1000
    df = spark.createDataFrame([("viento.lectura", 1, sensor, "Prueba", -15.5, -70.1, 50.0, 90.0, rafaga,
                                 10.0, 50.0, 640.0, t, "prueba-manual", t)], COLS)
    for _ in range(veces):
        (df.select(F.col("sensorId").alias("key"), to_json(struct(*COLS)).alias("value"))
           .write.format("kafka").option("kafka.bootstrap.servers", KAFKA).option("topic", TOPIC).save())

def tardio():
    time.sleep(20); publicar("viento-tardio", atraso_s=600); print(">>> publicado viento-tardio con 10 min de atraso")

threading.Thread(target=tardio, daemon=True).start()
conteo = (limpios.withWatermark("ts", "2 minutes").groupBy(F.window("ts", "1 minute"), "sensorId").count())
consulta = (conteo.writeStream.outputMode("update").format("console").option("truncate", "false")
            .option("checkpointLocation", f"{ART}/chk_s08_tardio").trigger(processingTime="10 seconds").start())

In [16]:
consulta.stop()
print("Si 'viento-tardio' NO aparece arriba, el watermark (2 min) lo descartó por llegar 10 min tarde.")

Si 'viento-tardio' NO aparece arriba, el watermark (2 min) lo descartó por llegar 10 min tarde.


## 7. **Deduplicación** con watermark — la usamos de verdad: (sensorId, obsTime)

In [17]:
dedup = (limpios.withWatermark("obs_ts", "30 minutes").dropDuplicates(["sensorId", "obsTime"]))
consulta = (dedup.select("sensorId", "obs_ts", "velocidadViento", "rafaga", "ts")
            .writeStream.outputMode("append").format("console").option("truncate", "false")
            .option("checkpointLocation", f"{ART}/chk_s08_dedup").trigger(processingTime="10 seconds").start())

In [18]:
time.sleep(5); publicar("viento-duplicado", atraso_s=0, rafaga=20.0, veces=2)
print(">>> 'viento-duplicado' publicado DOS veces con el mismo obsTime")

-------------------------------------------
Batch: 5
-------------------------------------------
+-----------------+-------------------+---------------+------+-----------------------+
|sensorId         |obs_ts             |velocidadViento|rafaga|ts                     |
+-----------------+-------------------+---------------+------+-----------------------+
|lampa-viento-03  |2026-11-02 19:30:00|2.4            |8.6   |2026-11-02 19:30:09.045|
|puno-viento-02   |2026-11-02 19:30:00|3.2            |22.0  |2026-11-02 19:30:09.045|
|juliaca-viento-01|2026-11-02 19:30:00|4.9            |19.8  |2026-11-02 19:30:09.045|
+-----------------+-------------------+---------------+------+-----------------------+



[Stage 20:=============>  (13 + 3) / 16][Stage 22:>                 (0 + 3) / 3]

>>> 'viento-duplicado' publicado DOS veces con el mismo obsTime


In [19]:
consulta.stop()
print("Debe aparecer una sola vez 'viento-duplicado', y cada sensor real una vez por cada paso de 15 min.")

-------------------------------------------
Batch: 6
-------------------------------------------
+--------+------+---------------+------+---+
|sensorId|obs_ts|velocidadViento|rafaga|ts |
+--------+------+---------------+------+---+
+--------+------+---------------+------+---+

Debe aparecer una sola vez 'viento-duplicado', y cada sensor real una vez por cada paso de 15 min.


## 8. Semántica de entrega y efecto del trigger en latencia/throughput

In [20]:
def medir(intervalo, segundos=30):
    q = (limpios.writeStream.outputMode("append").format("memory")
         .queryName("m_" + intervalo.replace(" ", "_")).trigger(processingTime=intervalo).start())
    q.awaitTermination(segundos); prog = q.recentProgress; q.stop()
    lotes = [p for p in prog if p["numInputRows"] > 0]
    dur = [p["durationMs"].get("triggerExecution", 0) for p in prog]
    return {"trigger": intervalo, "microlotes": len(prog), "lotes_con_datos": len(lotes),
            "filas": sum(p["numInputRows"] for p in prog), "ms_prom_por_lote": round(sum(dur) / max(len(dur), 1), 1)}

print(medir("2 seconds")); print(medir("10 seconds"))

>>> publicado viento-tardio con 10 min de atraso


{'trigger': '2 seconds', 'microlotes': 4, 'lotes_con_datos': 3, 'filas': 7, 'ms_prom_por_lote': 658.2}


{'trigger': '10 seconds', 'microlotes': 3, 'lotes_con_datos': 2, 'filas': 6, 'ms_prom_por_lote': 621.3}


**Semántica:** Kafka → Spark es *at-least-once* (si se cae entre procesar y guardar offset, se relee).
El sink Parquet + checkpoint lo vuelve *exactly-once* a nivel de archivos (los offsets y los archivos
se confirman juntos). La dedup por `(sensorId, obsTime)` cubre además los duplicados que **genera la fuente**.

### Umbrales operativos (para S09)
| Métrica | Umbral | Por qué |
|---|---|---|
| `inputRowsPerSecond` cae a 0 por > 5 min | alerta | el productor o la API se cayó |
| `triggerExecution` > intervalo del trigger | alerta | backpressure: Spark no alcanza |
| `avgOffsetsBehindLatest` > 100 | alerta | lag acumulado |
| % de inválidos > 5 % | alerta | cambió el contrato o la fuente |

In [21]:
consulta_parquet.stop()
for q in spark.streams.active: q.stop()